# Training Happy and Pessimistic LLMs Using PPO (RLHF)

In this project, we explore how to train two language models—one that generates **positive, cheerful responses (“Happy LLM”)** and another that produces **negative, pessimistic responses (“Pessimistic LLM”)**—to simulate different behavioral styles in customer service applications.

To achieve this, we use a reward function derived from a sentiment classifier trained on the IMDb movie review dataset. This classifier evaluates the sentiment of generated text and provides feedback signals that guide the learning process.

---

## Reinforcement Learning Overview

Reinforcement Learning (RL) is a branch of machine learning where an agent learns by interacting with an environment and receiving feedback in the form of rewards or penalties. The goal of the agent is to learn a policy that maximizes cumulative reward over time.

In this context, the **language model acts as the agent**, and its actions correspond to generating text tokens. Unlike supervised learning, RL does not require labeled input-output pairs. Instead, the model improves through exploration and feedback from the reward signal.

---

## Proximal Policy Optimization (PPO)

Proximal Policy Optimization (PPO) is a widely used reinforcement learning algorithm introduced by OpenAI. It is known for its balance between performance and stability.

PPO directly optimizes the policy while ensuring that updates remain controlled and do not deviate too much from the previous policy. This constraint helps maintain stable training and avoids destructive updates during learning.

---

## Project Goal

In this lab, you will learn how to implement and train a reinforcement learning agent using the PPO algorithm, focusing on sentiment-based text generation.

You will work with the IMDb dataset, which contains a large collection of movie reviews used for sentiment analysis tasks. By the end of this exercise, you will understand how PPO can be applied to fine-tune language models and how reinforcement learning techniques can be extended to other natural language processing problems.

---

### Import Libraries

In [3]:
import torch
import torch.nn as nn
import pandas as pd
from datasets import load_dataset
import trl
import json
from tqdm import tqdm
import matplotlib.pyplot as plt
from transformers import AutoTokenizer,AutoModelForCausalLM

helper functions

In [4]:
def load_from_json(filepath):
    
    with open (filepath, "r") as file:
        content = json.load(file)
    
    return content

In [5]:
def save_json_to_file(filepath, data):
    
    with open(filepath, "w") as file:
        json.dump(data, file, indent=4)
    
    print(f"{filepath} write sucessfully")
    

### Load and Config Dataset

In [6]:
dataset = load_dataset("imdb")
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [7]:
dataset_train = dataset['train']

In [8]:
for n in range(5):
    print("review: ",dataset_train['text'][n])
    print("label: ", dataset_train['label'][n])

review:  I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far bet